# Additional n=1 external validation prediction

to clean up and test

to check:

- only include ECoG and ipsilateral STN LFP
- exclude moments where was only Dyskinesia in body-side ipsilateral to ECoG (NOT CORRESPONDING WITH ECoG-hemisphere)

## Load packages and functions

In [ ]:
# Importing Python and external packages
import os
import sys
import importlib
import json
import csv
import pickle
from dataclasses import dataclass, field, fields
from itertools import compress
import pandas as pd
import numpy as np
from itertools import product
import sklearn as sk
from scipy import signal, stats

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns


In [ ]:
def get_project_path_in_notebook(
    subfolder: str = '',
):
    """
    Finds path of projectfolder from Notebook.
    Start running this once to correctly find
    other modules/functions
    """
    path = os.getcwd()

    while path[-20:] != 'dyskinesia_neurophys':

        path = os.path.dirname(path)
    
    return path

In [ ]:
# define local storage directories
projectpath = get_project_path_in_notebook()
codepath = os.path.join(projectpath, 'code')
figpath = os.path.join(projectpath, 'figures')
datapath = os.path.join(projectpath, 'data')
feat_path = os.path.join(projectpath, 'results', 'features')

In [ ]:
os.chdir(codepath)
# own utility functions
import utils.utils_fileManagement as utilsFiles
# own data exploration functions
import lfpecog_features.feats_read_proc_data as read_data
import lfpecog_preproc.preproc_import_scores_annotations as importClin
import lfpecog_analysis.ft_processing_helpers as ftProc
import lfpecog_analysis.import_ephys_results as importResults
import lfpecog_analysis.stats_fts_lid_corrs as ftLidCorr
import lfpecog_analysis.load_SSD_features as load_ssdFts
import lfpecog_analysis.ft_processing_helpers as ftProc
import lfpecog_features.feats_helper_funcs as ftHelp
from lfpecog_features.get_ssd_data import get_subject_SSDs
import lfpecog_predict.prepare_predict_arrays as prep_pred_arrs

from lfpecog_plotting.plotHelpers import get_colors
import lfpecog_plotting.plotHelpers as pltHelp
import lfpecog_plotting.plot_FreqCorr as plotFtCorrs
import lfpecog_plotting.plot_SSD_feat_descriptives as plot_ssd_descr

import lfpecog_analysis.get_acc_task_derivs as accDerivs

## 1) Define data, feature settings, and create DataClasses

In [ ]:
# set variables
DATA_VERSION = 'v4.0'    # v4.0: new artef-rem, no reref; v3.0 multiple re-ref
FT_VERSION = 'v8'  # v4: broad-flanks, bursts; v3: broad-flanked SSD
INCL_PSD_FTS=['mean_psd', 'variation']
IGNORE_PTS = ['011', '104', '106']

CDRS_RATER = 'Patricia'
ANALYSIS_SIDE = 'BILAT'
INCL_CORE_CDRS = True
CATEG_CDRS = False
MILD_CDRS = 4
SEV_CDRS = 8

INCL_ECOG = False
INCL_ACC = True

# path were classes are saved and loaded from
extVal_path = os.path.join(utilsFiles.get_project_path('data'),
                           'ext_val_prediction')

for debugging single sub feat classes

In [ ]:
importlib.reload(load_ssdFts)


# get all available subs with features
SUBS = utilsFiles.get_avail_ssd_subs(DATA_VERSION=DATA_VERSION,
                                     FT_VERSION=FT_VERSION,
                                     IGNORE_PTS=IGNORE_PTS)
print(f'SUBS: n={len(SUBS)} ({SUBS})')

# use as single ft example to debug/develop
sub_fts = load_ssdFts.ssdFeatures(
    sub_list=['023', '024'],
    settings_json=f'ftExtr_spectral_{FT_VERSION}.json',
)

In [ ]:
sub_fts.sub024

Prepare Class with FEATS and CDRS-LABELS

In [ ]:
# LOAD FEATURE via FeatureClass containing all features
importlib.reload(utilsFiles)
importlib.reload(accDerivs)
importlib.reload(ftProc)
importlib.reload(importClin)
importlib.reload(load_ssdFts)
importlib.reload(ftLidCorr)


FeatLid = ftProc.FeatLidClass(
    FT_VERSION=FT_VERSION,
    CDRS_RATER=CDRS_RATER,
    INCL_ECOG=INCL_ECOG,
    INCL_ACC_RMS=INCL_ACC,
    CATEGORICAL_CDRS=CATEG_CDRS,
    CORR_TARGET='CDRS',
    cutMild=MILD_CDRS, cutSevere=SEV_CDRS,
    TO_CALC_CORR=False,
    verbose=True,
)

# print(f'features included: {FEATS[sub].keys()}') 

Save Classes

In [ ]:
# SAVE FeatLabelClass as pickle

className = f'featLabels_n{len(FeatLid.FEATS.keys())}_ft{FT_VERSION}'
if FeatLid.CORR_TARGET == 'LID': className += '_Lid'
elif FeatLid.CATEGORICAL_CDRS == True: className += '_CatCdrs'
else: className += '_Cdrs'

if FeatLid.INCL_ECOG: className += '_Ecog'
else: className += '_StnOnly'

if INCL_ACC: className += '_ACC'

utilsFiles.save_class_pickle(class_to_save=FeatLid,
                             path=extVal_path,
                             filename=className)

Import classes

In [ ]:
# LOAD existing classes with features and labels

if INCL_ECOG:
    fname_ext = '_Ecog'
    n_subs = 14
else:
    fname_ext = '_StnOnly'
    n_subs = 22

if INCL_ACC: fname_ext += '_ACC'

predData = utilsFiles.load_class_pickle(
    os.path.join(extVal_path,
                 f'featLabels_n{n_subs}_ft{FT_VERSION}_Cdrs{fname_ext}.P'),
    convert_float_np64=True
)


## 2) Prepare prediction arrays

test:
- 1) all epochs
- 2) split epochs arbitrary based on movement (lineplot PPV/NPV)
- 2) A) only no-move epochs
- 2) B) only move epochs

Split training and external validation-test data

TODO: include here splitting on move or not

In [ ]:
dataDicts = {"train": {'FEATS': {}, 'LABELS': {}, 'ACC': {}},
             "extVal": {'FEATS': {}, 'LABELS': {}, 'ACC': {}}}

for sub in predData.FEATS.keys():
    if sub != '024':
        dataDicts['train']['FEATS'][sub] = predData.FEATS[sub]
        dataDicts['train']['LABELS'][sub] = predData.FT_LABELS[sub]
        dataDicts['train']['ACC'][sub] = predData.ACC_RMS[sub]

    elif sub == '024':
        dataDicts['extVal']['FEATS'][sub] = predData.FEATS[sub]
        dataDicts['extVal']['LABELS'][sub] = predData.FT_LABELS[sub]
        dataDicts['extVal']['ACC'][sub] = predData.ACC_RMS[sub]

print('n-train', len(dataDicts['train']['FEATS'].keys()),
      '; n-validate', len(dataDicts['extVal']['FEATS'].keys()))



In [ ]:

def merge_pred_dicts_to_grouparrays(dataDict, accDict=False,):

    list_returns = prep_pred_arrs.get_group_arrays_for_prediction(
        feat_dict=dataDict['FEATS'],
        label_dict=dataDict['LABELS'],
        CDRS_CODING='binary',  # categorical
        acc_dict=accDict,
    )
    if len(list_returns) == 6:
        (X_total, y_total_binary, y_total_scale,
         sub_ids_total, ft_times_total, ft_names) = list_returns
        acc_total = False  # set false bool bcs absent
    elif len(list_returns) == 7:
        (X_total, y_total_binary, y_total_scale,
         sub_ids_total, ft_times_total, ft_names,
         acc_total) = list_returns

    # Merge subject-arrays to one group array for prediction
    list_returns = prep_pred_arrs.merge_group_arrays(
        X_total=X_total,
        y_total_binary=y_total_binary,
        y_total_scale=y_total_scale,
        sub_ids_total=sub_ids_total,
        ft_times_total=ft_times_total,
        ext_acc_arr=acc_total
    )
    if len(list_returns) == 5:
        (X_all, y_all_binary, y_all_scale,
         sub_ids, ft_times_all) = list_returns
    elif len(list_returns) == 6:
        (X_all, y_all_binary, y_all_scale,
         sub_ids, ft_times_all, acc_total) = list_returns

    print(f'Subjects included ({len(np.unique(sub_ids))}): {np.unique(sub_ids)}')


    if accDict == False:
        return X_all, y_all_binary, sub_ids, ft_times_all, ft_names

    else:
        return X_all, y_all_binary, sub_ids, ft_times_all, ft_names, acc_total




In [ ]:
# Create arrays per subject based on features and labels

importlib.reload(prep_pred_arrs)

(X_all, y_all_binary,
 sub_ids, ft_times_all,
 ft_names, acc_all) = {}, {}, {}, {}, {}, {}

for label in dataDicts.keys():

    (
        X_all[label], y_all_binary[label],
        sub_ids[label], ft_times_all[label],
        ft_names[label], acc_all[label]
    ) = merge_pred_dicts_to_grouparrays(
        dataDicts[label], accDict=dataDicts[label]['ACC']
    )



In [ ]:
for label in dataDicts.keys():
    
    print(label, X_all[label].shape, y_all_binary[label].shape)

print(ft_names['extVal'])

In [ ]:
def convert_features(X_arr, ft_names):
    """
    takes average over bilat lfp features
    takes average over single gamma bands into gammaBroad
    """
    X_df = pd.DataFrame(X_arr, columns=ft_names)

    # change unilat into mean bilat LFP powers per band
    for band in ['theta', 'alpha', 'lo_beta', 'hi_beta',
                'gamma1', 'gamma2', 'gamma3', 'gammaPeak']:
        for ft in ['mean_psd', 'variation']:
            # add columns with data
            X_df[f'lfp_mean_{band}_{ft}'] = np.mean(
                [X_df[f'lfp_left_{band}_{ft}'],
                X_df[f'lfp_right_{band}_{ft}']], axis=0
            )
            # drop unilat lfp columns
            for s in ['left', 'right']: X_df = X_df.drop(labels=[f'lfp_{s}_{band}_{ft}'], axis=1)
        
        # drop imagniary coh columns
        X_df = X_df.drop(labels=[f'imag_coh_STN_STN_{band}'], axis=1)

    # take average over broad gamma
    # select all columns containing single gamma bands
    gamma_cols = [f for f in X_df.keys() if 'gamma1' in f]

    for col in gamma_cols:
        # add mean gammaBroad and drop single gamma cols
        X_df[col.replace('gamma1', 'gammaBroad')] = np.mean(
                [X_df[col],
                X_df[col.replace('gamma1', 'gamma2')],
                X_df[col.replace('gamma1', 'gamma3')]], axis=0
            )
        # drop unilat lfp columns
        X_df = X_df.drop(labels=[col,
                                    col.replace('gamma1', 'gamma2'),
                                    col.replace('gamma1', 'gamma3')], axis=1)
    
    return X_df.values, X_df.keys()

In [ ]:
for label in X_all.keys():
    X_all[label], ft_names[label] = convert_features(X_all[label], ft_names[label])

for label in dataDicts.keys():
    
    print(label, X_all[label].shape, y_all_binary[label].shape)

print(ft_names['extVal'])

Explore movement splitting based on clustering

- Cluster has issues to differentiate small movements and rest, for now pragmatic -0.5 as cut off

In [ ]:
# plt.hist(acc_all['extVal'], bins=np.arange(-1, 4, 0.05))

# plt.show()

In [ ]:
# from sklearn.cluster import KMeans
# from sklearn.mixture import GaussianMixture  # better for skewed data

# # # KMeans clustering into 2 groups
# # kmeans = KMeans(n_clusters=2, random_state=0)
# # k_labels = kmeans.fit_predict(acc_all['extVal'].reshape(-1, 1))

# # # Optional: identify which label corresponds to rest vs movement
# # # Rest is the cluster with the lower mean RMS
# # cluster_means = [acc_all['extVal'][k_labels == i].mean() for i in range(2)]
# # rest_label = np.argmin(cluster_means)
# # movement_label = 1 - rest_label


# # Fit Gaussian Mixture Model with 2 components
# rms_values = v
# # nonlinear transformation on rms bcs of skewedness of data
# rms_values = np.sign(rms_values) * (np.abs(rms_values) ** 1.5)

# gmm = GaussianMixture(n_components=3, random_state=0,)
# gmm_labels = gmm.fit_predict(rms_values.reshape(-1, 1))

# # Identify rest vs movement based on component means
# gmm_means = gmm.means_.flatten()
# rest_label = np.argmin(gmm_means)
# movement_label = 1 - rest_label

In [ ]:
# lab='extVal'
# plt.scatter(ft_times_all[lab], y_all_binary[lab],
#             s=10, alpha=.5, label='LID',)
# plt.scatter(ft_times_all[lab], acc_all[lab],
#             s=10, alpha=.5, label='acc-rms',)

# # plt.scatter(ft_times_all[lab], k_labels + 3,
# #             s=10, alpha=.3, label='k-cluster',)
# plt.scatter(ft_times_all[lab], gmm_labels + 3,
#             s=10, alpha=.3, label='cluster labels',)


# plt.legend()
# plt.show()

In [ ]:
# lab='train'
# plt.scatter(ft_times_all[lab], y_all_binary[lab],
#             s=10, alpha=.5, label='LID',)
# plt.scatter(ft_times_all[lab], acc_all[lab],
#             s=10, alpha=.3, label='acc-rms',)

# plt.legend()
# plt.show()

## 3) Prediction

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import gpboost as gpb

import joblib

# performance
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, plot_confusion_matrix, ConfusionMatrixDisplay,
    auc, roc_curve, RocCurveDisplay
)

In [ ]:
import lfpecog_predict.predict_helpers as predHelpers
import lfpecog_plotting.plot_pred_standards as plotPred

In [ ]:
ONLY_GAMMA = False

# Model name and Path to store or load
model_name = 'extVal_lda01'

if ONLY_GAMMA: model_name = 'extVal_lda02_onlyGamma'

model_path = os.path.join(extVal_path, f'{model_name}.pkl')


# INCLUDE ALL
X_train = X_all['train']
y_train = y_all_binary['train']

X_test = X_all['extVal']
y_true_test = y_all_binary['extVal']

times_test = ft_times_all['extVal']


# ONLY-GAMMA MODEL
if ONLY_GAMMA:
    gamma_idx = ['gamma' in f for f in ft_names['train']]  # select gamma features
    print(ft_names['extVal'][gamma_idx])
    X_train = X_train[:, gamma_idx]
    X_test = X_test[:, gamma_idx]


# Train LDA
lda = LDA()
lda.fit(X_train, y_train)

# # Save model to disk
# joblib.dump(lda, model_path)


In [ ]:
# Load saved model
lda_loaded = joblib.load(model_path)
print(f'Model: {model_path} loaded')


# Predict
y_pred = lda_loaded.predict(X_test)

# predict probabilities
y_proba = lda_loaded.predict_proba(X_test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                         gridspec_kw={'width_ratios': [2, 1]})

axes[0].scatter(times_test, y_pred, color='orange',
            label='LID binary preds',)
axes[0].scatter(times_test, y_proba[:, 0],
            color='green', s=10, alpha=.5,
            label='LID pred proba-1',)
axes[0].scatter(times_test, y_proba[:, 1],
            color='purple', s=10, alpha=.5,
            label='LID pred proba-2',)

axes[0].plot(times_test, y_true_test, c='gray',
         lw=5, alpha=.7, label='true binary LID',)


axes[0].set_xlabel('Time (min. vs LDOPA intake)')

axes[0].legend(ncol=2, bbox_to_anchor=[.01, 1.15],
           loc='upper left')


plt.show()

acc_score = accuracy_score(y_true=y_true_test, y_pred=y_pred)
print(f'Accuracy score: {np.round(acc_score, 3)}')

In [ ]:
# plot AUROC
fs = 14

fpr, tpr, _ = roc_curve(y_true_test, y_proba[:, 1])
auc_score = round(auc(fpr, tpr), 2)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.plot(fpr, tpr, c='darkgreen', lw=2,
        label=f'Prediction\n(AUC: {auc_score})',
)

ax.plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level (50/50)')

ax.set_xlabel('False Positive Rate', fontsize=fs, weight='bold',)
ax.set_ylabel('True Positive Rate', fontsize=fs, weight='bold',)
ax.set_title('Dyskinesia Prediction - Receiver Operator Curve'
            '\nLeave-One-Subject-Out cross-validation',
            fontsize=fs)

ax.legend(frameon=False, fontsize=fs, loc='lower right')
plt.tick_params(axis='both', labelsize=fs)
plt.tight_layout()
fname = f'extVal_Lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.show()

In [ ]:
plt.scatter(times_test, y_pred, color='orange',
            label='LID binary preds',)
plt.scatter(times_test, y_proba[:, 0],
            color='green', s=10, alpha=.5,
            label='LID pred proba-1',)
plt.scatter(times_test, y_proba[:, 1],
            color='purple', s=10, alpha=.5,
            label='LID pred proba-2',)

plt.plot(times_test, y_true_test, c='gray',
         lw=5, alpha=.7, label='true binary LID',)


plt.xlabel('Time (min. vs LDOPA intake)')

plt.legend(ncol=2, bbox_to_anchor=[.01, 1.15],
           loc='upper left')
plt.show()


acc_score = accuracy_score(y_true=y_true_test, y_pred=y_pred)
print(f'Accuracy score: {np.round(acc_score, 3)}')

In [ ]:
# plot AUROC
fs = 14

fpr, tpr, _ = roc_curve(y_true_test, y_proba[:, 1])
auc_score = round(auc(fpr, tpr), 2)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.plot(fpr, tpr, c='darkgreen', lw=2,
        label=f'Prediction\n(AUC: {auc_score})',
)

ax.plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level (50/50)')

ax.set_xlabel('False Positive Rate', fontsize=fs, weight='bold',)
ax.set_ylabel('True Positive Rate', fontsize=fs, weight='bold',)
ax.set_title('Dyskinesia Prediction - Receiver Operator Curve'
            '\nLeave-One-Subject-Out cross-validation',
            fontsize=fs)

ax.legend(frameon=False, fontsize=fs, loc='lower right')
plt.tick_params(axis='both', labelsize=fs)
plt.tight_layout()
fname = f'extVal_Lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.show()

Including move split

In [ ]:
ONLY_GAMMA = True

# SPLIT movement
MOVE_CUT = -.5
MOVE_SPLIT_train = acc_all['train'] > MOVE_CUT  # True for rel movement
MOVE_SPLIT_test = acc_all['extVal'] > MOVE_CUT  # True for rel movement

# Model A: no move
model_name = 'extVal_lda01rest'
if ONLY_GAMMA: model_name = 'extVal_lda02rest_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')  # Model Path to store or load

X_train_A = X_all['train'][~MOVE_SPLIT_train]
y_train_A = y_all_binary['train'][~MOVE_SPLIT_train]

X_test_A = X_all['extVal'][~MOVE_SPLIT_test]
y_true_test_A = y_all_binary['extVal'][~MOVE_SPLIT_test]
times_test_A = ft_times_all['extVal'][~MOVE_SPLIT_test]


# ONLY-GAMMA MODEL
if ONLY_GAMMA:
    gamma_idx = ['gamma' in f for f in ft_names['train']]  # select gamma features
    print(ft_names['extVal'][gamma_idx])
    X_train_A = X_train_A[:, gamma_idx]
    X_test_A = X_test_A[:, gamma_idx]



# Train LDA
lda = LDA()
lda.fit(X_train_A, y_train_A)

# Save model to disk
# joblib.dump(lda, model_path)



# Model B: movement
model_name = 'extVal_lda01move'
if ONLY_GAMMA: model_name = 'extVal_lda02move_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')  # Model Path to store or load

X_train_B = X_all['train'][MOVE_SPLIT_train]
y_train_B = y_all_binary['train'][MOVE_SPLIT_train]

X_test_B = X_all['extVal'][MOVE_SPLIT_test]
y_true_test_B = y_all_binary['extVal'][MOVE_SPLIT_test]
times_test_B = ft_times_all['extVal'][MOVE_SPLIT_test]


# ONLY-GAMMA MODEL
if ONLY_GAMMA:
    gamma_idx = ['gamma' in f for f in ft_names['train']]  # select gamma features
    print(ft_names['extVal'][gamma_idx])
    X_train_B = X_train_B[:, gamma_idx]
    X_test_B = X_test_B[:, gamma_idx]

# Train LDA
lda = LDA()
lda.fit(X_train_B, y_train_B)

# Save model to disk
# joblib.dump(lda, model_path)


In [ ]:
# check non move

# Load saved model
model_name = 'extVal_lda01rest'
if ONLY_GAMMA: model_name = 'extVal_lda02rest_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')
lda_loaded = joblib.load(model_path)


# Predict
y_pred_A = lda_loaded.predict(X_test_A)

# predict probabilities
y_proba_A = lda_loaded.predict_proba(X_test_A)

In [ ]:
# check rel move

# Load saved model
model_name = 'extVal_lda01move'
if ONLY_GAMMA: model_name = 'extVal_lda02move_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')
lda_loaded = joblib.load(model_path)


# Predict
y_pred_B = lda_loaded.predict(X_test_B)

# predict probabilities
y_proba_B = lda_loaded.predict_proba(X_test_B)

In [ ]:
# merge and sort rest and move arrays from two models
y_times_AB = np.concatenate([times_test_A, times_test_B])
t_sort_idx = np.argsort(y_times_AB)   # sort on chronological-times
merged_t = y_times_AB[t_sort_idx]

merged_y_pred = np.concatenate([y_pred_A, y_pred_B])[t_sort_idx]
merged_y_proba = np.concatenate([y_proba_A, y_proba_B])[t_sort_idx]
merged_y_true_test = np.concatenate([y_true_test_A, y_true_test_B])[t_sort_idx]



In [ ]:

plt.scatter(merged_t, merged_y_pred, color='orange',
            label='LID binary preds',)
plt.scatter(merged_t, merged_y_proba[:, 0],
            color='green', s=10, alpha=.5,
            label='LID pred proba-1',)
plt.scatter(merged_t, merged_y_proba[:, 1],
            color='purple', s=10, alpha=.5,
            label='LID pred proba-2',)

plt.plot(merged_t, merged_y_true_test, c='gray',
         lw=5, alpha=.7, label='true binary LID',)


plt.xlabel('Time (min. vs LDOPA intake)')

plt.legend(ncol=2, bbox_to_anchor=[.01, 1.15],
           loc='upper left')
plt.show()


acc_score = accuracy_score(y_true=merged_y_true_test, y_pred=merged_y_pred)
print(f'Accuracy score: {np.round(acc_score, 3)}')

In [ ]:
# plot AUROC
fs = 14

fpr, tpr, _ = roc_curve(merged_y_true_test, merged_y_proba[:, 1])
auc_score = round(auc(fpr, tpr), 2)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.plot(fpr, tpr, c='darkgreen', lw=2,
        label=f'Prediction\n(AUC: {auc_score})',
)

ax.plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level (50/50)')

ax.set_xlabel('False Positive Rate', fontsize=fs, weight='bold',)
ax.set_ylabel('True Positive Rate', fontsize=fs, weight='bold',)
ax.set_title('Dyskinesia Prediction - Receiver Operator Curve'
            '\nLeave-One-Subject-Out cross-validation',
            fontsize=fs)

ax.legend(frameon=False, fontsize=fs, loc='lower right')
plt.tick_params(axis='both', labelsize=fs)
plt.tight_layout()
fname = f'extVal_Lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.show()

In [ ]:

plt.scatter(merged_t, merged_y_pred, color='orange',
            label='LID binary preds',)
plt.scatter(merged_t, merged_y_proba[:, 0],
            color='green', s=10, alpha=.5,
            label='LID pred proba-1',)
plt.scatter(merged_t, merged_y_proba[:, 1],
            color='purple', s=10, alpha=.5,
            label='LID pred proba-2',)

plt.plot(merged_t, merged_y_true_test, c='gray',
         lw=5, alpha=.7, label='true binary LID',)


plt.xlabel('Time (min. vs LDOPA intake)')

plt.legend(ncol=2, bbox_to_anchor=[.01, 1.15],
           loc='upper left')
plt.show()

In [ ]:
# plot AUROC
fs = 14

fpr, tpr, _ = roc_curve(y_true_test, y_proba[:, 1])
auc_score = round(auc(fpr, tpr), 2)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.plot(fpr, tpr, c='darkgreen', lw=2,
        label=f'Prediction\n(AUC: {auc_score})',
)

ax.plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level (50/50)')

ax.set_xlabel('False Positive Rate', fontsize=fs, weight='bold',)
ax.set_ylabel('True Positive Rate', fontsize=fs, weight='bold',)
ax.set_title('Dyskinesia Prediction - Receiver Operator Curve'
            '\nLeave-One-Subject-Out cross-validation',
            fontsize=fs)

ax.legend(frameon=False, fontsize=fs, loc='lower right')
plt.tick_params(axis='both', labelsize=fs)
plt.tight_layout()
fname = f'extVal_Lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.show()

plot mean feature importances

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(18, 12))

fsize=20

sort_idx = np.argsort(np.mean(importances, axis=0))
temp_ftnames = plot_ssd_descr.readable_ftnames(ft_names)

ax.bar(np.arange(importances.shape[1]),
       np.mean(importances, axis=0)[sort_idx],)

ax.set_xticks(np.arange(len(ft_names)))
ax.set_xticklabels(np.array(temp_ftnames)[sort_idx],
                   rotation=60, ha='right', size=fsize)
ax.set_ylabel('importances (a.u.)', size=fsize + 8)
ax.set_xlabel('')

plt.tick_params(axis='both', size=fsize, labelsize=fsize+2)
plt.tight_layout()

fname = f'binaryLID_pred_ftImportances_ftsV4_lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.close()

Plot AUROC

In [ ]:
merged_y_pred.shape

In [ ]:
# plot AUROC
fs = 14

fpr, tpr, _ = roc_curve(merged_y_true_test, merged_y_proba[:, 1])
auc_score = round(auc(fpr, tpr), 2)

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.plot(fpr, tpr, c='darkgreen', lw=2,
        label=f'Prediction\n(AUC: {auc_score})',
)

ax.plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level (50/50)')

ax.set_xlabel('False Positive Rate', fontsize=fs, weight='bold',)
ax.set_ylabel('True Positive Rate', fontsize=fs, weight='bold',)
ax.set_title('Dyskinesia Prediction - Receiver Operator Curve'
            '\nLeave-One-Subject-Out cross-validation',
            fontsize=fs)

ax.legend(frameon=False, fontsize=fs, loc='lower right')
plt.tick_params(axis='both', labelsize=fs)
plt.tight_layout()
fname = f'extVal_Lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.show()


In [ ]:
importlib.reload(plotPred)

# Leave-One_subject-Out

# show metrics summary
print(classification_report(y_true_all, y_pred_all))

# show confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)
cm_figname = 'Group_LID_Pred_LDA_powCoh_confMatrix'
# plotPred.plot_confMatrix(cm, fig_path=figpath, fig_name=cm_figname,
#                          to_show=False, to_save=True)

# show Receiver Operator Cruve
fpr, tpr, _ = roc_curve(y_true_all, y_pred_conf_all,)
auc_score = auc(fpr, tpr)
acc_score = accuracy_score(y_true_all, y_pred_all)
print(f'AUC: {round(auc_score, 3)}, Accuracy: {round(acc_score, 3)}')
# roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr).plot()


Show individual prediction course

In [ ]:
importlib.reload(predHelpers)

# REAL PREDICTIONS returned per Subject
preds_subs, importances_sub = predHelpers.perform_prediction(
    X=X_all.copy(),
    y=y_all_binary.copy(),
    # y=y_all_scale.copy(),
    groups=sub_ids.ravel(),
    cv_method=LeaveOneGroupOut,
    clf_method='lda',
    perform_random_perm = False,
    n_perms = 0,
    verbose = False,
    return_dict_per_sub=True
)

In [ ]:
import lfpecog_plotting.plot_LID_predictions as plot_preds


In [ ]:
importlib.reload(plot_preds)

figname = f'Indiv_scaleLID_predict_lfpPows_{CDRS_RATER}'
figname += '_broadbSSD'

pred_fig_dir = os.path.join(figpath, 'prediction', ANALYSIS_SIDE.lower(),
                            f'ft_version_{FT_VERSION}',
                            f'n{len(SUBS)}')

plot_preds.plot_sub_gradual_preds(
    preds_subs=preds_subs, SUBS=SUBS,
    sub_ids=sub_ids,
    ft_times_all=ft_times_all,
    PLOT_FIG=True,
    SAVE_FIG=False,
    smooth_pred_samples=10,
    fig_name=figname, fig_dir=pred_fig_dir
)

In [ ]:

importlib.reload(plot_preds)

figname = f'Indiv_binLID_predict_v4Set2_{CDRS_RATER}'
figname += '_broadbSSD'

pred_fig_dir = os.path.join(figpath, 'prediction', ANALYSIS_SIDE.lower(),
                            f'ft_version_{FT_VERSION}',
                            f'n{len(SUBS)}')

plot_preds.plot_sub_binary_preds(
    preds_subs=preds_subs, SUBS=SUBS,
    sub_ids=sub_ids,
    y_all_scale=y_all_scale,
    ft_times_all=ft_times_all,
    PLOT_FIG=False,
    SAVE_FIG=False,
    fig_name=figname, fig_dir=pred_fig_dir)

#### Analyse Movement percentages

In [ ]:
importlib.reload(accDerivs)
accs, labels = {}, {}
for sub in FT_LABELS.keys():
    print(f'start sub {sub}')
    accs[sub], labels[sub] = accDerivs.load_acc_and_task(
        sub=sub, dataversion='v3.0', resample_freq=500)

Plot polar plot movement percentages in binary Groups

In [ ]:

ACT = {}

fig, axes = plt.subplots(len(FEATS.keys()), 1,
                         figsize=(8, len(FEATS.keys()) * 2))

for i_s, sub in enumerate(FEATS.keys()):

    sub_preds = preds_subs[sub]['pred']
    # if PLOT_PROBA: plot_probas = preds_subs[sub]['proba'][:, 1]
    # select labels and times for sub (included in prediction)
    sub_sel = sub_ids == sub
    sub_cdrs = y_all_scale[sub_sel]  # get CDRS as full scale
    sub_LID = y_all_binary[sub_sel]  # get binary LID
    sub_fttimes = ft_times_all[sub_sel]

    # get accelerometer info
    ecog_side = importClin.get_ecog_side(sub=sub)
    if ecog_side == 'right': body_side = 'left'
    elif ecog_side == 'left': body_side = 'right'

    acc_sub = []  # list to store

    for t in sub_fttimes:
        t = t * 60  # convert to seconds for acc-data
        idx_sel = np.logical_and(labels[sub].index.values > t,
                                 labels[sub].index.values < (t + WIN_LEN_sec))
        act = labels[sub][idx_sel][[f'{body_side}_tap', f'{body_side}_move']]
        acc_sub.append(sum(np.max(act, axis=1).values) / act.shape[0] * 100)

    ACT[sub] = np.array(acc_sub)

    assert len(ACT[sub]) == len(sub_LID) == len(sub_preds), (
        f'ACC ({len(ACT[sub])}), feat lengths ({len(sub_LID)})'
        f', and pred lengths ({len(sub_preds)}) not equal')
    
    axes[i_s].plot(acc_sub, label='activitiy %')
    axes[i_s].fill_between(x=np.arange(len(sub_LID)), y1=0, y2=10,
                     label='true LID binary',
                     where=sub_LID, color='orange', alpha=.5,)
    axes[i_s].set_title(sub)
    axes[i_s].set_ylabel('unilateral activitiy-%')
    axes[i_s].set_xlabel(f'{WIN_LEN_sec}s-windows')
    axes[i_s].legend()
plt.tight_layout()

figname = 'indivMovement_vs_BinaryLID'
plt.savefig(os.path.join(figpath, 'prediction', figname),
        dpi=300, facecolor='w',)

plt.close()

In [ ]:
clrs = list(get_colors().values())

total_act_prc = {'Dyskinesia ABSENT': {'all': [],
                                       'predicted present': [],
                                       'predicted absent': []},
                 'Dyskinesia PRESENT': {'all': [],
                                       'predicted present': [],
                                       'predicted absent': []}}

fig, axes = plt.subplots(len(FEATS.keys()), 2,
                         figsize=(8, len(FEATS.keys()) * 3))

for i_sub, sub in enumerate(FEATS.keys()):

    for LID_BIN, LID_NAME  in enumerate(['Dyskinesia ABSENT', 'Dyskinesia PRESENT']):
        sub_sel = sub_ids == sub
        sub_LID = y_all_binary[sub_sel]  # get TRUE binary LID
        true_lid_mask = sub_LID == LID_BIN
        act_only_TRUE_LID_sel = ACT[sub][true_lid_mask]
        
        axes[i_sub, LID_BIN].hist(act_only_TRUE_LID_sel,
                           label=f'all true {LID_NAME}',
                           color=clrs[0], alpha=.3,)
        total_act_prc[LID_NAME]['all'].extend(act_only_TRUE_LID_sel)  # store in total dict

        sub_preds = preds_subs[sub]['pred']  # get PREDICTED binary labels
        preds_only_TRUE_LID_sel = sub_preds[true_lid_mask]

        
        pred_mask = preds_only_TRUE_LID_sel == 1

        if True in pred_mask:

            axes[i_sub, LID_BIN].hist(act_only_TRUE_LID_sel[~pred_mask],
                            label=f'no-LID-predicted',
                            color=clrs[5], alpha=.5, align='left',)
            total_act_prc[LID_NAME]['predicted absent'].extend(
                act_only_TRUE_LID_sel[~pred_mask])  # store in total dict

        if False in pred_mask:

            axes[i_sub, LID_BIN].hist(act_only_TRUE_LID_sel[pred_mask],
                            label=f'LID-predicted',
                            color=clrs[2], alpha=.5, align='right',)
            total_act_prc[LID_NAME]['predicted present'].extend(
                act_only_TRUE_LID_sel[pred_mask])  # store in total dict
                
        axes[i_sub, LID_BIN].set_title(f'sub-{sub}: expert-rated {LID_NAME}')
        axes[i_sub, LID_BIN].set_ylabel('observations')
        axes[i_sub, LID_BIN].set_xlabel('Activity per window (%)')
        axes[i_sub, LID_BIN].legend()
    
plt.tight_layout()  


figname = 'binaryLID_pred_INDIVmovementDistribution'
plt.savefig(os.path.join(figpath, 'prediction', figname),
            dpi=300, facecolor='w',)

plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for LID_BIN, LID_NAME  in enumerate(['Dyskinesia ABSENT', 'Dyskinesia PRESENT']):
    
    axes[LID_BIN].hist(total_act_prc[LID_NAME]['all'],
                        label=f'all true {LID_NAME}',
                        color=clrs[0], alpha=.3,)
    
    for i2, PRED_NAME in enumerate(['predicted absent', 'predicted present']):
        aligns = ['left', 'right']
        axes[LID_BIN].hist(total_act_prc[LID_NAME][PRED_NAME],
                            label=PRED_NAME,
                            color=clrs[5-i2*3], alpha=.5, align=aligns[i2],)
    
    axes[LID_BIN].set_title(f'expert-rated {LID_NAME}')
    axes[LID_BIN].set_ylabel('observations')
    axes[LID_BIN].set_xlabel('Activity per window (%)')
    axes[LID_BIN].legend()

plt.close()

Polar plot for mvoement distribution during LID and true/false predictions

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(16, 8),
                       subplot_kw={"projection": "polar"},
                       )

fontsize = 14
false_colors = np.array(clrs)[[0, 2]]
correct_colors = np.array(clrs)[[3, 5]]


for LID_BIN, TRUE_LID_NAME  in enumerate(['Dyskinesia ABSENT',
                                          'Dyskinesia PRESENT']):
    
    if 'present' in TRUE_LID_NAME.lower():
        preds_correct = total_act_prc[TRUE_LID_NAME]['predicted present']
        preds_false = total_act_prc[TRUE_LID_NAME]['predicted absent']
    elif 'absent' in TRUE_LID_NAME.lower():
        preds_correct = total_act_prc[TRUE_LID_NAME]['predicted absent']
        preds_false = total_act_prc[TRUE_LID_NAME]['predicted present']
    else:
        raise ValueError('no present absent found')

    n_bins = len(preds_correct) + len(preds_false)

    ANGLES = np.linspace(0, 2 * np.pi, n_bins, endpoint=False) + (np.pi/2)
    WIDTH = 2 * np.pi / n_bins

    ACT_PRCS = list(preds_correct) + list(preds_false)

    axes[LID_BIN].bar(x=ANGLES[:len(preds_correct)],
                      height=np.array(preds_correct) + 5, #bottom=-10,
                    #   height=sorted(np.array(preds_correct) + 5, reverse=True), #bottom=-10,
                      color=false_colors[LID_BIN], alpha=0.8,
                      width=WIDTH,
                      label=f'preds correct ({round(len(preds_correct)/n_bins*100)}%)')
    axes[LID_BIN].bar(x=ANGLES[len(preds_correct):],
                      height=np.array(preds_false) + 5, #bottom=-10,  # plus 5 to show zeros
                    #   height=sorted(np.array(preds_false) + 5), #bottom=-10,  # plus 5 to show zeros
                      color=correct_colors[LID_BIN], alpha=0.8,
                      width=WIDTH,
                      label=f'preds false ({round(len(preds_false)/n_bins*100)} %)')

    axes[LID_BIN].set_ylim(0, 35)
    axes[LID_BIN].set_yticks(np.arange(0, 31, 5))
    axes[LID_BIN].set_yticklabels([' '] + [f'{y}%' for y in np.arange(0, 26, 5)],
                                  fontsize=fontsize)
    axes[LID_BIN].set_xticks([])
    axes[LID_BIN].set_xticklabels([], )

    axes[LID_BIN].set_title(f'{TRUE_LID_NAME} (expert-rated)',
                            fontsize=fontsize + 4, weight='bold')
    axes[LID_BIN].set_ylabel(f'Activity per {WIN_LEN_sec}s-window (%)',
                             fontsize=fontsize + 4)
    axes[LID_BIN].legend(fontsize=fontsize + 4,
                         frameon=False, ncol=2, loc='upper center',
                         bbox_to_anchor=(.5, -.05))

    print('plotted', TRUE_LID_NAME)

figname = 'binaryLID_pred_movementDistribution'
plt.savefig(os.path.join(figpath, 'prediction', figname),
            dpi=300, facecolor='w',)
plt.close()